# DIESEL — Notebook 05: Final Evaluation & Comparison
Three-way comparison: Base (zero-shot) vs Round 1 vs Round 2 (TGDA).
Generates all publication tables, figures, and statistical tests.

**Runtime:** Select the **L4 GPU** kernel.

**Prerequisites:** All previous notebooks completed.

In [ ]:
!pip install -q torch transformers peft trl bitsandbytes
!pip install -q accelerate datasets sqlparse sqlglot
!pip install -q wandb matplotlib seaborn scipy scikit-learn python-dotenv tqdm

In [ ]:
from huggingface_hub import login
login()

In [ ]:
# === Colab Setup: Clone project & set path ===
import os, sys

if not os.path.exists('/content/Text-to-SQL-LLM/src'):
    !git clone https://github.com/Deepak-Lingala/Text-to-SQL-LLM.git /content/Text-to-SQL-LLM

os.chdir('/content/Text-to-SQL-LLM')
sys.path.insert(0, '/content/Text-to-SQL-LLM')
print('Project root:', os.getcwd())

import torch, gc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
ROUND1_ADAPTER = 'outputs/round1/final_adapter'
ROUND2_ADAPTER = 'outputs/round2/final_adapter'
SPIDER_DB_DIR = 'spider_databases'
MAX_SAMPLES = None  # None = full eval

In [ ]:
from src.config import get_config
from src.data_loader import load_spider_dataset
from src.model_loader import load_finetuned_model, load_base_model, load_tokenizer
from src.evaluate import evaluate_model, compare_models
from src.error_analyzer import analyze_errors, compare_error_distributions, ErrorCategory

config = get_config()
sns.set_theme(style='whitegrid', font_scale=1.1)
FIGURES_DIR = config.paths.figures_dir
os.makedirs(FIGURES_DIR, exist_ok=True)

dataset, schema_manager = load_spider_dataset(config)
eval_data = dataset['validation']
if MAX_SAMPLES:
    eval_data = eval_data.select(range(min(MAX_SAMPLES, len(eval_data))))
print(f'Evaluation examples: {len(eval_data)}')

## Evaluate All 3 Models

In [ ]:
all_results = []
all_analyses = []

models_to_eval = [
    ('base_zero_shot', None),
    ('round1_finetuned', ROUND1_ADAPTER),
    ('round2_tgda', ROUND2_ADAPTER),
]

for model_name, adapter_path in models_to_eval:
    print(f'\n{"="*60}')
    print(f'Evaluating: {model_name}')
    print(f'{"="*60}')
    
    if adapter_path:
        model, tokenizer = load_finetuned_model(adapter_path, config)
    else:
        model = load_base_model(config)
        tokenizer = load_tokenizer(config)
        model.eval()
    
    results = evaluate_model(
        model=model, tokenizer=tokenizer,
        eval_data=eval_data, schema_manager=schema_manager,
        spider_db_dir=SPIDER_DB_DIR, config=config,
        model_name=model_name, save_dir=config.paths.eval_dir,
    )
    all_results.append(results)
    
    analysis = analyze_errors(
        eval_results=results,
        schema_map=schema_manager.schema_map,
        config=config,
    )
    all_analyses.append(analysis)
    
    del model; gc.collect(); torch.cuda.empty_cache()

## Statistical Comparison

In [ ]:
comparison = compare_models(all_results, save_dir=config.paths.eval_dir)

# Error distribution shifts
if len(all_analyses) >= 2:
    shift_r1 = compare_error_distributions(all_analyses[0], all_analyses[1], save_dir=config.paths.error_analysis_dir)
if len(all_analyses) >= 3:
    shift_r2 = compare_error_distributions(all_analyses[1], all_analyses[2], save_dir=config.paths.error_analysis_dir)

## Publication Figures

In [ ]:
# Figure 8: Three-Way Accuracy Comparison
fig, ax = plt.subplots(figsize=(10, 6))
model_names = [r['model_name'] for r in all_results]
accuracies = [r['overall_accuracy'] * 100 for r in all_results]
ci_lower = [r.get('confidence_interval', {}).get('lower_95', 0) * 100 for r in all_results]
ci_upper = [r.get('confidence_interval', {}).get('upper_95', 0) * 100 for r in all_results]
err_lower = [a - l for a, l in zip(accuracies, ci_lower)]
err_upper = [u - a for a, u in zip(accuracies, ci_upper)]

colors = ['#FF7043', '#42A5F5', '#66BB6A']
bars = ax.bar(model_names, accuracies, color=colors, edgecolor='gray', linewidth=0.5)
ax.errorbar(model_names, accuracies, yerr=[err_lower, err_upper],
            fmt='none', color='black', capsize=5, capthick=2)
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)
ax.set_ylabel('Execution Accuracy (%)')
ax.set_title('DIESEL: Three-Way Model Comparison on Spider Dev')
ax.set_ylim(0, max(accuracies) + 15)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig8_three_way_comparison.png'), bbox_inches='tight')
plt.show()

In [ ]:
# Figure 9: Difficulty-Stratified Comparison
fig, ax = plt.subplots(figsize=(12, 7))
all_diffs = set()
for r in all_results:
    all_diffs.update(r.get('difficulty_breakdown', {}).keys())
difficulties = sorted(all_diffs)
x = np.arange(len(difficulties))
width = 0.25

for idx, (res, color) in enumerate(zip(all_results, colors)):
    accs = [res.get('difficulty_breakdown', {}).get(d, {}).get('accuracy', 0) * 100 for d in difficulties]
    ax.bar(x + idx * width, accs, width, label=res['model_name'], color=color, alpha=0.85)

ax.set_xlabel('Query Difficulty')
ax.set_ylabel('Execution Accuracy (%)')
ax.set_title('Accuracy by Difficulty Level')
ax.set_xticks(x + width)
ax.set_xticklabels(difficulties)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig9_difficulty_comparison.png'), bbox_inches='tight')
plt.show()

In [ ]:
# Figure 10: Error Distribution Shift (3 models)
cats = ErrorCategory.ALL
cat_labels = [ErrorCategory.LABELS[c] for c in cats]
fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(cats))
width = 0.25

for idx, (analysis, color, name) in enumerate(zip(
    all_analyses, colors, [r['model_name'] for r in all_results]
)):
    pcts = [analysis['category_distribution'][c]['percentage_of_errors'] for c in cats]
    ax.bar(x + idx * width, pcts, width, label=name, color=color, alpha=0.85)

ax.set_xlabel('Error Category')
ax.set_ylabel('Percentage of Errors (%)')
ax.set_title('Error Category Distribution: Base → Round 1 → Round 2 (TGDA)')
ax.set_xticks(x + width)
ax.set_xticklabels(cat_labels, rotation=30, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig10_error_shift_3way.png'), bbox_inches='tight')
plt.show()

## Results Summary

In [ ]:
print('\n' + '=' * 60)
print('DIESEL — Final Evaluation Complete')
print('=' * 60)
for res in all_results:
    ci = res.get('confidence_interval', {})
    name = res['model_name']
    acc = res['overall_accuracy'] * 100
    lo = ci.get('lower_95', 0) * 100
    hi = ci.get('upper_95', 0) * 100
    print(f'  {name:25s}: {acc:.1f}% (95% CI: [{lo:.1f}%, {hi:.1f}%])')

print(f'\nAll figures saved to: {FIGURES_DIR}')
print(f'Results saved to: {config.paths.eval_dir}')